In [1]:
from qdrant_client import QdrantClient
from qdrant_client.models import (
    VectorParams,
    Distance,
    PointStruct
)
from sentence_transformers import SentenceTransformer
from langchain_text_splitters import MarkdownHeaderTextSplitter
from pathlib import Path
import markdown
from bs4 import BeautifulSoup
import uuid

In [2]:
client = QdrantClient(
    url="YOUR_URL",
    api_key="YOUR_API_KEY"
)
print(client.get_collections())

COLLECTION_NAME = "KnowledgeBase"

collections=[CollectionDescription(name='KnowledgeBase')]


In [3]:
embedding_model = SentenceTransformer("BAAI/bge-small-en-v1.5") # for External RAG

In [4]:
file_path = "KB.md"

md_text = Path(file_path).read_text(
    encoding="utf-8"
)


In [5]:
headers_to_split_on = [
    ("#", "main_topic")
]

markdown_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=headers_to_split_on
)

docs = markdown_splitter.split_text(md_text)

print(f"Total semantic topic chunks: {len(docs)}")

# Preview chunks
for i, doc in enumerate(docs[:3]):
    print("=" * 80)
    print("METADATA:", doc.metadata)
    print(doc.page_content[:500])
    print() 

Total semantic topic chunks: 51
METADATA: {'main_topic': 'User Details'}
My name is rohan Agrawal aand i am a CEO in vought International.

METADATA: {'main_topic': 'Artificial Intelligence (AI)'}
Artificial Intelligence (AI) refers to the simulation of human intelligence processes by computer systems. It encompasses the ability of machines to learn from experience, adapt to new inputs, and perform human-like tasks. Core components include reasoning, problem-solving, perception, and linguistic intelligence. AI spans various subfields, from simple rule-based systems to complex neural networks, aiming to automate tasks and enhance decision-making across numerous industries.

METADATA: {'main_topic': 'Machine Learning (ML)'}
Machine Learning (ML) is a subset of AI that focuses on building systems capable of learning from data without explicit programming. It relies on statistical algorithms to identify patterns, make predictions, and improve performance over time as more data becomes avai

In [6]:
# 8. GENERATE EMBEDDINGS
# =========================
texts = [doc.page_content for doc in docs]

embeddings = embedding_model.encode(
    texts,
    show_progress_bar=True,
    normalize_embeddings=True
)



Batches:   0%|          | 0/2 [00:00<?, ?it/s]

In [7]:
points = []

for idx, (doc, embedding) in enumerate(zip(docs, embeddings)):

    points.append(
        PointStruct(
            id=str(uuid.uuid4()),
            vector=embedding.tolist(),
            payload={
                "chunk_id": idx,
                "text": doc.page_content,
                "metadata": doc.metadata,
                "source": file_path
            }
        )
    )


In [8]:
# 10. UPSERT TO QDRANT
# =========================

client.upsert(
    collection_name=COLLECTION_NAME,
    points=points
)


UpdateResult(operation_id=1, status=<UpdateStatus.COMPLETED: 'completed'>)

In [10]:
# 11. TEST RETRIEVAL
# =========================

query = "User name is?"

query_embedding = embedding_model.encode(
    query,
    normalize_embeddings=True
)

results = client.search(
    collection_name=COLLECTION_NAME,
    query_vector=query_embedding.tolist(),
    limit=3
)

print("\nTop Matches:\n")

for result in results:

    print("=" * 80)
    print("Score:", result.score)
    print(result.payload["text"])
    print()


Top Matches:

Score: 0.54682857
Natural Language Processing (NLP) is an AI discipline concerned with the interaction between computers and human language. It enables machines to read, understand, interpret, and generate human language in a valuable way. Techniques involve tokenization, sentiment analysis, named entity recognition, and sequence-to-sequence modeling. NLP drives technologies such as chatbots, translation services, and text summarization, bridging the gap between human communication and digital understanding.

Score: 0.543791
My name is rohan Agrawal aand i am a CEO in vought International.

Score: 0.5317453
Artificial Intelligence (AI) refers to the simulation of human intelligence processes by computer systems. It encompasses the ability of machines to learn from experience, adapt to new inputs, and perform human-like tasks. Core components include reasoning, problem-solving, perception, and linguistic intelligence. AI spans various subfields, from simple rule-based sys